# Part 3: Building a RAG Chatbot

This guide provides instructions for the final step in our project: transforming your conversational chatbot into a full Retrieval-Augmented Generation (RAG) application. This will give our chatbot a "memory" in the form of external documents, allowing it to answer questions about information it was never trained on.

> This notebook is a guide to enhancing your project. **Do not run the code cells here directly.** Follow the instructions in your local development environment.

-----

## 1.&nbsp; Introduction to RAG

Our current chatbot is conversational, but its knowledge is limited to what the base LLM was trained on. It has a knowledge cut-off and cannot answer questions about your private documents or recent events.

RAG gives our LLM a "library" of documents to read from before it answers a question. The RAG process involves three key stages:

1.  Indexing: We process our documents into a searchable knowledge library. This is a one-time, upfront process.
2.  Retrieval: When you ask a question, the system first finds the most relevant snippets of text from the library.
3.  Generation: The LLM receives both your original question and the relevant snippets it retrieved. It then uses this information to generate a factual, context-aware answer.

-----

## 2.&nbsp; Configuration and Data

First, we need to provide our RAG system with a document to learn from.

### 2.1 Add the Data

1. In the root directory create a new directory called `data`.
2. Download the plain text version of "[Alice's Adventures in Wonderland](https://www.gutenberg.org/ebooks/11.txt.utf-8)" from Project Gutenberg
3. Right-click on the Alice in Wonderland page and save it file as `alice_in_wonderland.txt` inside your `data` folder.

### 2.2 Update the Configuration

Next, we need to add new parameters to our configuration file to manage the embedding model and the RAG process.

Open `src/config.py` and add the following, making sure the import is at the top of the file:

In [ ]:
from pathlib import Path


# --- All of your ---
# --- other settings ---


# --- Embedding Model Configuration ---
EMBEDDING_MODEL_NAME: str = "sentence-transformers/all-MiniLM-L6-v2"


# --- RAG/VectorStore Configuration ---
# The number of most relevant text chunks to retrieve from the vector store
SIMILARITY_TOP_K: int = 2
# The size of each text chunk in tokens
CHUNK_SIZE: int = 512
# The overlap between adjacent text chunks in tokens
CHUNK_OVERLAP: int = 50


# --- Chat Memory Configuration ---
CHAT_MEMORY_TOKEN_LIMIT: int = 3900


# --- Persistent Storage Paths (using pathlib for robust path handling) ---
ROOT_PATH: Path = Path(__file__).parent.parent
DATA_PATH: Path = ROOT_PATH / "data/"
EMBEDDING_CACHE_PATH: Path = ROOT_PATH / "local_storage/embedding_model/"
VECTOR_STORE_PATH: Path = ROOT_PATH / "local_storage/vector_store/"


We've added three new sections to our configuration file to manage the RAG pipeline.
* Embedding Model: This section specifies the `EMBEDDING_MODEL_NAME`, which is the model responsible for turning our text into numerical representations (vectors) so the system can understand its meaning.
* RAG/VectorStore: Here, we control how our documents are processed. This involves chunking, which means breaking large documents into smaller, digestible pieces. We define this with `CHUNK_SIZE` and `CHUNK_OVERLAP`. To get a better intuition for how these two parameters work together, this [visualisation tool](https://chunkviz.up.railway.app/) is very helpful. We also set `SIMILARITY_TOP_K`, which tells the engine how many of the most relevant chunks to retrieve when answering a question.
* Paths: Finally, we've set up robust paths for storing our data and cached files. You'll notice we're using Python's `pathlib` library. This is a modern standard for handling file paths because it automatically resolves differences between operating systems. For example, Windows uses backslashes (`C:\\Users\\...`) while macOS and Linux use forward slashes (`/Users/...`). `pathlib` handles this for us, ensuring our code is portable and works anywhere without modification.

-----

## 3.&nbsp; The Embedding Model

An embedding model is a specialised model that converts text into a list of numbers, called a vector. It's trained to ensure that semantically similar pieces of text have mathematically similar vectors. This is the key to how our system can find the "most relevant" parts of the book.

Let's add a function to `model_loader.py` (our "parts store") to initialise this new component.

1.  Open `src/model_loader.py`.
2.  Add the new imports and `get_embedding_model` function to the file.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from src.config import (
    EMBEDDING_MODEL_NAME,
    EMBEDDING_CACHE_PATH,
)

def get_embedding_model() -> HuggingFaceEmbedding:
    """Initialises and returns the HuggingFace embedding model"""

    # Create the cache directory if it doesn't exist
    EMBEDDING_CACHE_PATH.mkdir(parents=True, exist_ok=True)

    return HuggingFaceEmbedding(
        model_name=EMBEDDING_MODEL_NAME,
        cache_folder=EMBEDDING_CACHE_PATH.as_posix()
    )


> `.as_posix()` is a method from Python's pathlib library that converts a file path object into a string.

-----

## 4.&nbsp; Building the RAG Engine

Now for the most significant upgrade. We'll completely overhaul `engine.py` to build our RAG pipeline. We'll do this by adding functions one by one, explaining the purpose of each as we go.

First, open `src/engine.py` and delete all existing content. We'll start with a clean slate.

### 4.1 The Imports

Let's begin by adding all the necessary imports to the top of our empty `src/engine.py` file. This brings all the tools we'll need into our script.

In [ ]:
from llama_index.core import (
    StorageContext,
    SimpleDirectoryReader,
    VectorStoreIndex,
    load_index_from_storage,
)
from llama_index.core.chat_engine.types import BaseChatEngine
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq

from src.config import (
    CHUNK_OVERLAP,
    CHUNK_SIZE,
    DATA_PATH,
    LLM_SYSTEM_PROMPT,
    SIMILARITY_TOP_K,
    VECTOR_STORE_PATH,
    CHAT_MEMORY_TOKEN_LIMIT,
)
from src.model_loader import (
    get_embedding_model,
    initialise_llm
)

### 4.2 Step 1: Creating the Vector Store

The first piece of our RAG pipeline is the indexing process. We need a function that can read our documents, break them into chunks, and convert them into a searchable vector store.

Add the following function to your `src/engine.py` file.

In [ ]:
def _create_new_vector_store(
        embed_model: HuggingFaceEmbedding
) -> VectorStoreIndex:
    """Creates, saves, and returns a new vector store from documents."""

    print(
        "Creating new vector store from all files in the 'data' directory..."
    )

    # This reads all the text files in the specified directory.
    documents: list[Document] = SimpleDirectoryReader(
        input_dir=DATA_PATH
    ).load_data()

    if not documents:
        raise ValueError(
            f"No documents found in {DATA_PATH}. Cannot create vector store."
        )

    # This breaks the long document into smaller, more manageable chunks.
    text_splitter: SentenceSplitter = SentenceSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )

    # This is the core of the vector store. It takes the text chunks,
    # uses the embedding model to convert them to vectors, and stores them.
    index: VectorStoreIndex = VectorStoreIndex.from_documents(
        documents,
        transformations=[text_splitter],
        embed_model=embed_model
    )

    # This saves the newly created index to disk for future use.
    index.storage_context.persist(persist_dir=VECTOR_STORE_PATH.as_posix())
    print("Vector store created and saved.")
    return index

  * `_create_new_vector_store`: The underscore `_` at the beginning is a Python convention. It signals to other developers that this is an "internal" or "private" helper function. It's not meant to be called directly from outside this file.
  * `SimpleDirectoryReader`: This is a convenient LlamaIndex tool that automatically finds and reads all supported files (like `.txt`) from a given directory.
  * `SentenceSplitter`: We can't feed an entire corpus to the LLM at once due to context window limits. The splitter breaks the document into chunks based on the `CHUNK_SIZE` and `CHUNK_OVERLAP` we set in our config.
  * `VectorStoreIndex.from_documents`: It orchestrates the creation of the vector store by taking our documents and using the provided `embed_model` to create the vector embeddings.
  * `.persist()`: Indexing can be slow and computationally expensive. By saving the result to disk, we can simply load it next time instead of rebuilding it.

### 4.3 Step 2: Loading the Vector Store

Now we need a function that can either load our pre-built index from disk or, if it doesn't exist, call our internal helper function to create it. This function will be the main public entry point for accessing our knowledge base.

Add the following function below the previous one in `src/engine.py`.

In [ ]:
def get_vector_store(embed_model: HuggingFaceEmbedding) -> VectorStoreIndex:
    """
    Loads the vector store from disk if it exists;
    otherwise, creates a new one.
    """

    # Create the parent directory if it doesn't exist.
    VECTOR_STORE_PATH.mkdir(parents=True, exist_ok=True)

    # Check if the directory contains any files.
    if any(VECTOR_STORE_PATH.iterdir()):
        print("Loading existing vector store from disk...")
        storage_context: StorageContext = StorageContext.from_defaults(
            persist_dir=VECTOR_STORE_PATH.as_posix()
        )
        # We must provide the embed_model when loading the index.
        return load_index_from_storage(
            storage_context,
            embed_model=embed_model
        )
    else:
        # If the directory is empty,
        # call our internal function to build the index.
        return _create_new_vector_store(embed_model)

  * `mkdir(parents=True, exist_ok=True)`: This is a safe way to ensure our storage path exists. `parents=True` creates any necessary parent folders, and `exist_ok=True` prevents an error if the folder already exists.
  * `any(VECTOR_STORE_PATH.iterdir())`: This is a clever way to check if a directory is empty. This check specifically asks, "Are there any files inside this folder?"
  * `load_index_from_storage`: This LlamaIndex function loads the persisted index from disk. It's crucial that we pass the `embed_model` here. The stored index only contains the document vectors; to answer a question, we need the model to turn the question into a vector, and it must be the exact same model to ensure the comparison is valid.

### 4.4 Step 3: Assembling the RAG Chat Engine

With our data pipeline complete, we can now create the function that assembles the final chat engine. It will take the LLM and embedding model as inputs and use our `get_vector_store` function to connect to the knowledge base.

Add the following function to `src/engine.py`.

In [ ]:
def get_chat_engine(
        llm: Groq,
        embed_model: HuggingFaceEmbedding
) -> BaseChatEngine:
    """Initialises and returns the main conversational RAG chat engine."""

    vector_index: VectorStoreIndex = get_vector_store(embed_model)
    memory: ChatMemoryBuffer = ChatMemoryBuffer.from_defaults(
        token_limit=CHAT_MEMORY_TOKEN_LIMIT
    )

    # Assemble the RAG chat engine
    chat_engine: BaseChatEngine = vector_index.as_chat_engine(
        memory=memory,
        llm=llm,
        system_prompt=LLM_SYSTEM_PROMPT,
        similarity_top_k=SIMILARITY_TOP_K,
    )
    return chat_engine

  * `vector_index.as_chat_engine`: This is a high-level LlamaIndex method that builds a chat engine directly from a vector index.

### 4.5 Step 4: The Main Application Loop

Finally, we update our main loop. Its job is now simply to initialise the components from `components.py` and pass them to our new engine factory function.

Add this final function to `src/engine.py`.

In [ ]:
def main_chat_loop() -> None:
    """Main application loop to run the RAG chatbot."""

    print("--- Initialising models... ---")
    llm: Groq = initialise_llm()
    embed_model: HuggingFaceEmbedding = get_embedding_model()

    chat_engine: BaseChatEngine = get_chat_engine(
        llm=llm,
        embed_model=embed_model
    )
    print("--- RAG Chatbot Initialised. ---")
    chat_engine.chat_repl()


Your `engine.py` file is now complete! It cleanly separates the logic for creating, loading, and using your knowledge base to power a fully-featured RAG chatbot.

-----

## 5.&nbsp; Run the RAG Chatbot

With all the new components in place, you're ready to run the final application.

From your VSCode terminal, run the `main.py` file:

In [ ]:
python main.py

The first time you run this it might take a minute or two, as it will create the vector store. Every time you run the application after that, it will say "Loading existing vector store..." and start almost instantly.

Now, ask the chatbot a question: "What did the Dormouse say in Alice in Wonderland?"

It'll retrieve the relevant passages from the book and provide a correct, factual answer.

-----

## 6.&nbsp; The Full RAG Journey

Now that you have a working RAG chatbot, let's trace the complete journey of a single question through our system:

1.  You type a question (e.g., "What did the Dormouse say?") and press Enter.
2.  The `chat_repl()` in `engine.py` captures your input.
3.  The `chat_engine` receives the question. Because it's in `"context"` mode, it first needs to retrieve relevant documents.
4.  The engine uses the `embed_model` (from `components.py`) to convert your question string into a query vector.
5.  This query vector is sent to the `vector_index` (loaded by `get_vector_store`).
6.  The index performs a semantic search, comparing your query vector to all the document chunk vectors it stores. It finds the `top_k` (e.g., 4) most similar chunks.
7.  These relevant text chunks are taken and inserted into the `system_prompt`, along with your original question.
8.  This new, combined prompt (context + question) is sent to the `llm` (from `components.py`).
9.  The LLM reads the context and the question and generates a final, factually-grounded answer.
10. The `chat_repl()` prints the final answer to your screen.

-----

## 7.&nbsp; Challenges and Further Exploration 😀

You've now successfully built a complete, conversational RAG-powered chatbot. It can learn from your documents, remember your conversation, and run efficiently by loading a persistent knowledge base. You can now experiment by adding different documents to the `data` folder and exploring the different configuration settings in `src/config.py`.

### 7.1. Personalise the Knowledge Base

**How to Change the Data:**

1.  Open your project in VSCode.
2.  Navigate to the `data` directory and delete the `alice_in_wonderland.txt` file.
3.  Add your own documents to the `data` directory. For best results, start with 2-3 plain text or PDF files related to a single topic.
4.  Navigate to the project's root folder and delete the entire `local_storage` directory. This is crucial as it forces the system to create a new knowledge base from your documents the next time it runs.

**Challenge:**
* Run the application (`python main.py`). The first start will be slow as it builds the new index.
* Ask the chatbot several questions that can only be answered using your new documents to ensure it's working correctly.
* Have a few conversations to get an initial impression. Does it sound coherent, or are some answers a bit "crazy"? Take note of any strange responses that you can investigate in the next challenge.

### 7.2. Experiment with Different Embedding Models

The embedding model is the core component that converts your text into meaningful numerical representations, enabling the chatbot to find relevant information. Changing the model can dramatically alter the quality of the retrieval and, consequently, the final answers.

**How to Find and Choose a New Model:**

1.  Navigate to the Leaderboard: Go to the [Hugging Face MTEB Leaderboard](https://huggingface.co/spaces/mteb/leaderboard). This is a central hub for comparing the performance of hundreds of embedding models.
2.  Customise the Benchmark: Below the main plot, find and click on "customise this benchmark". Since this is a retrieval project, select "Retrieval" from the "Task(s)" dropdown. Feel free to research any other settings you are unsure of to better tailor the benchmark to your needs.
3.  Read the Table and Understand its Impact: The table shows key metrics for each model. Here's what they mean for your RAG application:
    * Retrieval (Mean (TaskType)): This is the most important score for our project. It measures how well the model finds relevant information. A higher score suggests the model will do a better job of retrieving the correct text chunks from your documents, leading to more accurate and coherent answers from the chatbot.
    * Number of Parameters: This affects performance and quality. A model with more parameters might have a more nuanced understanding of language, potentially improving retrieval. However, it will be slower to download, require more disk space, and use more memory (RAM), making the initial indexing of your documents take longer. This is a classic trade-off between quality and speed.
    * Embedding Dimensions: This impacts the size and detail of your vector store. A larger dimension (e.g., 4096) can capture more detail about the text, but it will make your saved index in the `local_storage/vector_store` folder larger. Searching through these larger vectors can also be slightly slower.
    * Max Tokens: This is the maximum length of a text chunk the model can process. This is important in relation to the `CHUNK_SIZE` you set in `src/config.py`. Your `CHUNK_SIZE` must be smaller than the model's `Max Tokens` limit (which is high for most models, e.g., 32768), otherwise you will get errors during indexing.

**How to Change the Embedding Model in Your Project:**

1.  From the leaderboard, click on the name of a model you want to try.
2.  On the model's page, copy its ID, which is displayed in the top-left corner (e.g., `Qwen/Qwen1.5-Embedding-8B`).
3.  Open the `src/config.py` file in your project.
4.  Update the `EMBEDDING_MODEL_NAME` variable with the new model ID you copied.
5.  Crucially, delete the `local_storage` directory again. The existing index was built with the old model and is incompatible with the new one. The application must re-index your documents with the new model's understanding of language.

**Challenge:**
* Try at least two new models. Choose one with a very high retrieval score and perhaps a much smaller one (fewer parameters) to compare the trade-offs.
* For each new model, run the application and ask it the same complex questions, especially the ones that gave you "crazy" answers before.
* What to look for: Pay close attention to the relevance of the answers.
    * A good embedding will lead to answers that are highly relevant, accurate, and directly address your questions, even nuanced ones. The chatbot will feel "smarter" because it's finding the correct source material.
    * A bad or mismatched embedding will cause the chatbot to retrieve irrelevant information. This results in answers that are off-topic, nonsensical, or fail to use the knowledge in your documents, even when you know it's there.